# Credit Card Fraud Detection

Exploring the classic imbalanced classification problem using the Kaggle credit card fraud dataset.

284,807 transactions, 492 fraud cases (0.17%). The goal is to catch fraud without drowning in false positives.

> Run `python data/download.py` first to get `data/creditcard.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, roc_auc_score,
    precision_recall_curve, average_precision_score
)
import xgboost as xgb
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## Load Data

In [ ]:
df = pd.read_csv('data/creditcard.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
df.info()
df.isnull().sum().sum()  # should be 0

## Class Imbalance

This is the core problem. 99.83% of transactions are legitimate.

In [ ]:
counts = df['Class'].value_counts()
print(counts)
print(f"\nFraud rate: {counts[1] / len(df) * 100:.3f}%")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['Legit', 'Fraud'], counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Class Distribution (count)')
axes[0].set_ylabel('Count')

axes[1].bar(['Legit', 'Fraud'], counts.values / counts.sum() * 100, color=['steelblue', 'tomato'])
axes[1].set_title('Class Distribution (%)')
axes[1].set_ylabel('Percent')
plt.tight_layout()
plt.show()

## Feature Distributions

V1-V28 are PCA components (anonymized). Only `Amount` and `Time` are raw features.

In [ ]:
fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(legit['Amount'], bins=50, alpha=0.6, label='Legit', color='steelblue', density=True)
axes[0].hist(fraud['Amount'], bins=50, alpha=0.6, label='Fraud', color='tomato', density=True)
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Density')
axes[0].set_title('Transaction Amount by Class')
axes[0].set_xlim(0, 500)
axes[0].legend()

axes[1].hist(legit['Time'] / 3600, bins=48, alpha=0.6, label='Legit', color='steelblue', density=True)
axes[1].hist(fraud['Time'] / 3600, bins=48, alpha=0.6, label='Fraud', color='tomato', density=True)
axes[1].set_xlabel('Hours elapsed')
axes[1].set_ylabel('Density')
axes[1].set_title('Transaction Time by Class')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Check which PCA features show the most separation
v_cols = [f'V{i}' for i in range(1, 15)]
fig, axes = plt.subplots(2, 7, figsize=(18, 6))
for i, feat in enumerate(v_cols):
    ax = axes[i // 7][i % 7]
    ax.hist(legit[feat], bins=40, alpha=0.5, color='steelblue', density=True)
    ax.hist(fraud[feat], bins=40, alpha=0.5, color='tomato', density=True)
    ax.set_title(feat, fontsize=9)
    ax.set_yticks([])
plt.suptitle('V1-V14: Fraud (red) vs Legit (blue)', y=1.01)
plt.tight_layout()
plt.show()

## Preprocessing

Scale `Amount` and `Time` (V features are already PCA-normalized), then stratified train/test split.

In [ ]:
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled'] = scaler.fit_transform(df[['Time']])

feature_cols = [f'V{i}' for i in range(1, 29)] + ['Amount_scaled', 'Time_scaled']
X = df[feature_cols]
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Train fraud: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Test fraud:  {y_test.sum()} ({y_test.mean()*100:.2f}%)")

## Evaluate Helper

In [ ]:
def evaluate(name, model, threshold=0.5):
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= threshold).astype(int)
    print(f"\n--- {name} (threshold={threshold:.2f}) ---")
    print(classification_report(y_test, preds, target_names=['Legit', 'Fraud'], digits=4))
    roc = roc_auc_score(y_test, proba)
    ap = average_precision_score(y_test, proba)
    print(f"ROC-AUC:  {roc:.4f}  |  Avg Precision: {ap:.4f}")
    return proba

## Approach 1: Baseline XGBoost

No resampling, no class weights. Standard XGBoost on a 99.8%/0.2% split.

In [ ]:
model_baseline = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    eval_metric='logloss', random_state=42, n_jobs=-1
)
model_baseline.fit(X_train, y_train)
proba_baseline = evaluate('Baseline XGBoost', model_baseline)

## Approach 2: SMOTE

Oversample the minority class in the training set to 50/50 balance.

In [ ]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"After SMOTE - shape: {X_train_sm.shape}")
print(f"Class counts: {dict(pd.Series(y_train_sm).value_counts())}")

model_smote = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    eval_metric='logloss', random_state=42, n_jobs=-1
)
model_smote.fit(X_train_sm, y_train_sm)
proba_smote = evaluate('SMOTE + XGBoost', model_smote)

## Approach 3: scale_pos_weight

Tell XGBoost directly how much to penalize missing fraud cases.

In [ ]:
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {ratio:.1f}")

model_weighted = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=ratio, eval_metric='logloss',
    random_state=42, n_jobs=-1
)
model_weighted.fit(X_train, y_train)
proba_weighted = evaluate('Class Weights XGBoost', model_weighted)

## Approach 4: Threshold Tuning

Train with class weights, then find the threshold that maximizes F1 on fraud.

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_test, proba_weighted)
f1s = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx = f1s[:-1].argmax()  # last element has no threshold
best_thresh = thresholds[best_idx]
print(f"Best threshold: {best_thresh:.3f}  |  F1 at that point: {f1s[best_idx]:.4f}")

proba_tuned = evaluate('Threshold Tuned (weighted model)', model_weighted, threshold=best_thresh)

## Precision-Recall Curves

ROC-AUC looks great for all models because the negative class is huge. PR curves show what actually matters.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, proba in [('Baseline', proba_baseline), ('SMOTE', proba_smote), ('Class Weights', proba_weighted)]:
    p, r, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    ax.plot(r, p, label=f'{name} (AP={ap:.3f})')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves')
ax.legend()
plt.tight_layout()
plt.savefig('precision_recall_curves.png', dpi=150)
plt.show()

## Feature Importance

In [ ]:
imp = pd.Series(model_weighted.feature_importances_, index=feature_cols).nlargest(10)
plt.figure(figsize=(8, 5))
imp.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 10 Features - Class Weights Model')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## Summary

| Approach | Fraud Recall | Fraud Precision | Notes |
|---|---|---|---|
| Baseline | low | high | misses most fraud |
| SMOTE | higher | lower | more false alarms |
| Class Weights | high | medium | good balance |
| Threshold Tuned | tunable | tunable | best control |

**Key takeaway:** standard accuracy is useless on this dataset. The class weights + threshold tuning approach gives the best control over the precision-recall trade-off without introducing synthetic data leakage from SMOTE.